In [12]:
#Source : https://github.com/PacktPublishing/Python-Natural-Language-Processing-Cookbook-Second-Edition/tree/main/data

In [13]:
# !python -m spacy download en_core_web_lg
import nltk
import spacy
from nltk import word_tokenize
from nltk.corpus import stopwords
from string import punctuation
small_model = spacy.load("en_core_web_sm")
large_model = spacy.load("en_core_web_lg")
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/lwhitenack/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/lwhitenack/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Finding triplets using spaCy

In [14]:
sentences = [
    "The big black cat stared at the small dog.",
    "Jane watched her brother in the evenings.",
    "Nick was driving to Madrid."
]
verb_patterns = [
    [{"POS": "VERB"}],
    [{"POS": "VERB"}, {"POS": "ADP"}],
    [{"POS": "AUX", "OP": "?"}, {"POS": "VERB"}, {"POS": "ADP", "OP": "?"}]
]

In [15]:
from spacy.matcher import Matcher
matcher = Matcher(small_model.vocab)
matcher.add("VP", verb_patterns)

In [16]:
def find_verb_phrase(doc, matcher):
    matches = matcher(doc)
    verb_phrases = [match for match in matches if small_model.vocab.strings[match[0]] == "VP"]
    verb_phrase_spans = [doc[match[1]:match[2]] for match in verb_phrases]
    verb_phrase_spans.sort(key=len, reverse=True)
    verb_phrase = verb_phrase_spans[0]
    root = verb_phrase[0]
    for token in verb_phrase:
        if token.dep_ == "ROOT":
            root = token
    return verb_phrase, root

In [17]:
def get_subject_phrase(doc):
    for token in doc:
        if ("subj" in token.dep_):
            subtree = list(token.subtree)
            start = subtree[0].i
            end = subtree[-1].i + 1
            return doc[start:end]

In [18]:
def get_object_phrase(doc):
    for token in doc:
        if ("dobj" in token.dep_):
            subtree = list(token.subtree)
            start = subtree[0].i
            end = subtree[-1].i + 1
            return doc[start:end]

In [19]:
def get_dative_phrase(doc):
    for token in doc:
        if ("dative" in token.dep_):
            subtree = list(token.subtree)
            start = subtree[0].i
            end = subtree[-1].i + 1
            return doc[start:end]

In [20]:
def get_prepositional_phrase_objs(doc):
    prep_spans = []
    for token in doc:
        if ("pobj" in token.dep_):
            subtree = list(token.subtree)
            start = subtree[0].i
            end = subtree[-1].i + 1
            prep_spans.append(doc[start:end])
    return prep_spans

In [21]:
for sentence in sentences:
    doc = small_model(sentence)
    verb_phrase, root = find_verb_phrase(doc, matcher)
    subject_phrase = get_subject_phrase(doc)
    object_phrase = get_object_phrase(doc)
    prep_phrases = get_prepositional_phrase_objs(doc)
    if object_phrase is None:
        object_phrase = prep_phrases[0]
    print(subject_phrase, "\t", verb_phrase, "\t", object_phrase)

The big black cat 	 stared at 	 the small dog
Jane 	 watched 	 her brother
Nick 	 was driving to 	 Madrid


# Finding triplets using GPT

In [22]:
import openai
openai.api_key = "test"

In [23]:
prompt="""Find subject, verb, object triplets in the following sentence.
Create a python dictionary structure of the form: {"subject": Subject, "verb": Verb, "object": Object}
Sentence: Nick was driving to Madrid."""
response = openai.Completion.create(
    model="text-davinci-003",
    prompt=prompt,
    temperature=0,
    max_tokens=256,
    top_p=1.0,
    frequency_penalty=0,
    presence_penalty=0
)
print(response)

APIRemovedInV1: 

You tried to access openai.Completion, but this is no longer supported in openai>=1.0.0 - see the README at https://github.com/openai/openai-python for the API.

You can run `openai migrate` to automatically upgrade your codebase to use the 1.0.0 interface. 

Alternatively, you can pin your installation to the old version, e.g. `pip install openai==0.28`

A detailed migration guide is available here: https://github.com/openai/openai-python/discussions/742
